# 02 · Exploratory and statistical analysis

This notebook uses the **training part only**. Validation and test blocks stay unseen, so nothing learned here can leak into model choice or the final score.

**It creates** the tables in `reports/tables/` (descriptive statistics, event means, absence shares, correlations, logistic coefficients, k-means cluster rates) and the figures `event_label_correlation.png`, `event_correlation_heatmap.png`, `logistic_event_coefficients.png`, `pca_scree_plot.png`, `pca_scatter.png`, `kmeans_elbow.png`.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if not (root / 'datasets').exists():
    root = root.parent
raw = root / 'datasets' / 'hdfs_v1' / 'preprocessed'
processed = root / 'datasets' / 'processed'
figures = root / 'reports' / 'figures'
tables = root / 'reports' / 'tables'
tables.mkdir(parents=True, exist_ok=True)

NORMAL_COLOUR = '#8a8a8a'
ANOMALY_COLOUR = '#d4591c'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

In [2]:
df = pd.read_csv(processed / 'hdfs_clean.csv')
splits = np.load(processed / 'splits.npz')
train = df.iloc[splits['train_idx']]
event_columns = [name for name in df.columns if name.startswith('E') and name[1:].isdigit() and name != 'E22']
templates = pd.read_csv(raw / 'HDFS.log_templates.csv').set_index('EventId')['EventTemplate']

print(train.shape)
print(train['target'].value_counts())

(345035, 31)
target
0    334933
1     10102
Name: count, dtype: int64


## Descriptive statistics by class

Mean, median, standard deviation, minimum and maximum of each event count, for normal and anomalous training blocks.

In [3]:
normal = train.loc[train['target'] == 0, event_columns]
anomaly = train.loc[train['target'] == 1, event_columns]
normal_stats = normal.agg(['mean', 'median', 'std', 'min', 'max']).T
anomaly_stats = anomaly.agg(['mean', 'median', 'std', 'min', 'max']).T
display(normal_stats)
display(anomaly_stats)

,mean,median,std,min,max
E1,0.000000,0.0,0.000000,0.0,0.0
E2,0.074821,0.0,0.315802,0.0,4.0
E3,0.294999,0.0,2.422456,0.0,203.0
E4,0.303455,0.0,1.721390,0.0,41.0
E5,3.007437,3.0,0.128081,3.0,12.0
E6,0.007434,0.0,0.127883,0.0,9.0
E7,0.000000,0.0,0.000000,0.0,0.0
E8,0.000000,0.0,0.000000,0.0,0.0
E9,3.000000,3.0,0.000000,3.0,3.0
E10,0.000000,0.0,0.000000,0.0,0.0


,mean,median,std,min,max
E1,0.000594,0.0,0.034462,0.0,2.0
E2,0.282419,0.0,0.682656,0.0,4.0
E3,1.103643,0.0,4.478291,0.0,203.0
E4,0.768165,0.0,1.602673,0.0,15.0
E5,2.851713,3.0,1.354056,1.0,13.0
E6,0.429915,0.0,0.678774,0.0,10.0
E7,0.017026,0.0,0.215035,0.0,5.0
E8,0.004257,0.0,0.068080,0.0,2.0
E9,2.110572,3.0,1.367936,0.0,3.0
E10,0.009206,0.0,0.130546,0.0,3.0


In [4]:
# Save both classes in one tidy table: one row per event and class.
normal_stats['class'] = 'normal'
anomaly_stats['class'] = 'anomaly'
descriptive = pd.concat([normal_stats, anomaly_stats]).rename_axis('event').reset_index()
descriptive.to_csv(tables / 'training_descriptive_statistics.csv', index=False)

## Mean counts and how often each event is absent

`absent` is the share of blocks where the event never happened. Several events never appear in a normal training block, so even a single occurrence is suspicious.

In [5]:
means = pd.DataFrame(index=event_columns)
means['normal_mean'] = normal.mean()
means['anomaly_mean'] = anomaly.mean()
means['difference'] = means['anomaly_mean'] - means['normal_mean']
means['normal_absent'] = (normal == 0).mean()
means['anomaly_absent'] = (anomaly == 0).mean()
display(means.sort_values('difference', ascending=False))

,normal_mean,anomaly_mean,difference,normal_absent,anomaly_absent
E3,0.294999,1.103643,0.808644,0.921363,0.664126
E4,0.303455,0.768165,0.464710,0.932163,0.704415
E6,0.007434,0.429915,0.422481,0.995029,0.666304
E18,0.007234,0.427242,0.420008,0.995029,0.664522
E25,0.007234,0.427242,0.420008,0.995029,0.664522
E16,0.007234,0.422788,0.415553,0.995029,0.666304
E2,0.074821,0.282419,0.207598,0.938519,0.816175
E20,0.000657,0.200257,0.199601,0.999352,0.816670
E13,0.000000,0.132746,0.132746,1.000000,0.955751
E28,0.000030,0.115918,0.115888,0.999970,0.886656


In [6]:
means[['normal_mean', 'anomaly_mean', 'difference']].rename_axis('event').to_csv(tables / 'event_mean_counts.csv')
means[['normal_absent', 'anomaly_absent']].rename_axis('event').to_csv(tables / 'event_absence_share.csv')

never_in_normal = means.index[(means['normal_absent'] == 1) & (means['anomaly_absent'] < 1)]
print('events never seen in a normal training block but seen in anomalies:', list(never_in_normal))

events never seen in a normal training block but seen in anomalies: ['E1', 'E7', 'E8', 'E10', 'E12', 'E13', 'E14', 'E15', 'E17', 'E19', 'E24', 'E29']


## Correlation with the anomaly label

Pearson correlation between each event count and the 0/1 target. Events that never vary (always the same count) have no correlation and are dropped.

In [7]:
varying_events = [name for name in event_columns if train[name].std() > 0]
constant_events = [name for name in event_columns if name not in varying_events]
print('constant in training data:', constant_events)

corr = train[varying_events + ['target']].corr()['target'].drop('target').sort_values()
corr_table = pd.DataFrame({'correlation_with_anomaly': corr, 'template': templates[corr.index]})
corr_table.rename_axis('event').sort_values('correlation_with_anomaly', key=abs, ascending=False).to_csv(
    tables / 'event_correlations.csv')

constant in training data: []


In [8]:
fig, ax = plt.subplots(figsize=(6, 7))
colours = [ANOMALY_COLOUR if value > 0 else NORMAL_COLOUR for value in corr.values]
ax.barh(corr.index, corr.values, color=colours)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with anomaly label (training blocks)')
ax.set_title('Event correlation with anomaly label')
fig.tight_layout()
fig.savefig(figures / 'event_label_correlation.png', dpi=160)
plt.show()

/tmp/ipykernel_10702/2398894234.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The template text explains what the strongest events are.

In [9]:
top_events = corr.abs().sort_values(ascending=False).head(6).index
display(corr_table.loc[top_events])

,correlation_with_anomaly,template
E9,-0.539436,[*]Received block[*]of size[*]from[*]
E11,-0.532016,[*]PacketResponder[*]for block[*]terminating[*]
E25,0.398280,[*]BLOCK* ask[*]to replicate[*]to[*]
E18,0.398280,[*]Starting thread to transfer block[*]to[*]
E16,0.395934,[*]:Transmitted block[*]to[*]
E20,0.389106,[*]Unexpected error trying to delete block[*]B...


## Correlation between events

Some events always happen together (for example, one event per replica). Strongly correlated features carry the same information, which matters for logistic regression coefficients below.

In [10]:
event_corr = train[varying_events].corr()

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(event_corr.values, cmap='RdGy_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(varying_events)), varying_events, rotation=90)
ax.set_yticks(range(len(varying_events)), varying_events)
fig.colorbar(image, ax=ax, label='Pearson correlation')
ax.set_title('Correlation between event counts (training blocks)')
fig.tight_layout()
fig.savefig(figures / 'event_correlation_heatmap.png', dpi=160)
plt.show()

/tmp/ipykernel_10702/2989989969.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# List the pairs that move almost perfectly together.
pairs = []
for i, first in enumerate(varying_events):
    for second in varying_events[i + 1:]:
        if abs(event_corr.loc[first, second]) > 0.95:
            pairs.append((first, second, round(event_corr.loc[first, second], 4)))
print(pairs)

[('E5', 'E26', np.float64(0.9612)), ('E6', 'E16', np.float64(0.9872)), ('E6', 'E18', np.float64(0.985)), ('E6', 'E25', np.float64(0.985)), ('E7', 'E14', np.float64(0.964)), ('E9', 'E11', np.float64(0.9928)), ('E10', 'E14', np.float64(0.9644)), ('E16', 'E18', np.float64(0.9978)), ('E16', 'E25', np.float64(0.9978)), ('E18', 'E25', np.float64(1.0))]


## Logistic regression coefficients

Counts are standardised first, so coefficients are comparable. A positive coefficient raises the anomaly score when the other counts are held fixed.
Because some events are almost perfectly correlated (above), individual coefficients can split or swap sign between twin events; read them as a ranking, not as causes.

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train[event_columns])
y_train = train['target']

logistic = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logistic.fit(X_train, y_train)

coef = pd.DataFrame({'event': event_columns, 'coefficient': logistic.coef_[0]})
coef['absolute_coefficient'] = coef['coefficient'].abs()
coef['template'] = templates[coef['event']].values
coef = coef.sort_values('absolute_coefficient', ascending=False)
coef.to_csv(tables / 'logistic_event_coefficients.csv', index=False)
display(coef)

,event,coefficient,absolute_coefficient,template
20,E21,7.260113,7.260113,[*]Deleting block[*]file[*]
15,E16,-6.790754,6.790754,[*]:Transmitted block[*]to[*]
17,E18,3.577611,3.577611,[*]Starting thread to transfer block[*]to[*]
23,E25,3.577611,3.577611,[*]BLOCK* ask[*]to replicate[*]to[*]
24,E26,-3.143616,3.143616,[*]BLOCK* NameSystem[*]addStoredBlock: blockMa...
21,E23,-3.006316,3.006316,[*]BLOCK* NameSystem[*]delete:[*]is added to i...
25,E27,2.378299,2.378299,[*]BLOCK* NameSystem[*]addStoredBlock: Redunda...
12,E13,1.839954,1.839954,[*]Receiving empty packet for block[*]
5,E6,1.725686,1.725686,[*]Received block[*]src:[*]dest:[*]of size[*]
8,E9,-1.598576,1.598576,[*]Received block[*]of size[*]from[*]


In [13]:
top_coef = coef.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(6, 5))
colours = [ANOMALY_COLOUR if value > 0 else NORMAL_COLOUR for value in top_coef['coefficient']]
ax.barh(top_coef['event'], top_coef['coefficient'], color=colours)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised coefficient (balanced logistic regression)')
ax.set_title('Twelve largest logistic coefficients')
fig.tight_layout()
fig.savefig(figures / 'logistic_event_coefficients.png', dpi=160)
plt.show()

/tmp/ipykernel_10702/2326691656.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## PCA

PCA finds the directions along which event counts vary most. The scree plot shows how much variance each component explains; the scatter shows a fixed random sample of training blocks on the first two components.

In [14]:
pca = PCA(random_state=42)
components = pca.fit_transform(X_train)
cumulative = np.cumsum(pca.explained_variance_ratio_)
print('components needed for 95% of the variance:', np.argmax(cumulative >= 0.95) + 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(cumulative) + 1), pca.explained_variance_ratio_, marker='o', color='black')
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance ratio')
ax.set_title('PCA scree plot (standardised training counts)')
fig.tight_layout()
fig.savefig(figures / 'pca_scree_plot.png', dpi=160)
plt.show()

components needed for 95% of the variance: 16


/tmp/ipykernel_10702/2030505377.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Saved for the website's scree chart.
pca_table = pd.DataFrame({'component': range(1, len(cumulative) + 1),
                          'explained_variance_ratio': pca.explained_variance_ratio_,
                          'cumulative': cumulative})
pca_table.to_csv(tables / 'pca_explained_variance.csv', index=False)

In [16]:
# A fixed sample keeps the scatter readable; anomalies are drawn last so they are not hidden.
rng = np.random.default_rng(42)
sample = rng.choice(len(components), size=8000, replace=False)
sample_target = y_train.values[sample]

fig, ax = plt.subplots(figsize=(6, 5))
for value, colour, name in [(0, NORMAL_COLOUR, 'Normal'), (1, ANOMALY_COLOUR, 'Anomaly')]:
    points = components[sample][sample_target == value]
    ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.6, color=colour, label=name)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('First two PCA components (8,000 training blocks)')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(figures / 'pca_scatter.png', dpi=160)
plt.show()

/tmp/ipykernel_10702/282420563.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## k-means on normal behaviour

k-means describes a fixed sample of normal training blocks. The elbow is gradual rather than decisive; k=4 is a modest illustrative choice, not a proven optimum or an anomaly classifier. Class labels are used only to describe the resulting training clusters.


In [17]:
normal_X = scaler.transform(normal)
normal_sample = normal_X[rng.choice(len(normal_X), size=30000, replace=False)]

inertias = []
k_values = range(1, 11)
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(normal_sample)
    inertias.append(model.inertia_)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(list(k_values), inertias, marker='o', color='black')
ax.set_xlabel('Number of clusters k')
ax.set_ylabel('Inertia')
ax.set_title('k-means elbow (normal training sample)')
fig.tight_layout()
fig.savefig(figures / 'kmeans_elbow.png', dpi=160)
plt.show()

/tmp/ipykernel_10702/951068475.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Saved for the website's elbow chart.
pd.DataFrame({'k': list(k_values), 'inertia': inertias}).to_csv(tables / 'kmeans_elbow.csv', index=False)

In [19]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(normal_sample)
clusters = kmeans.predict(X_train)

cluster_table = pd.DataFrame({'cluster': clusters, 'target': y_train.values})
cluster_rates = cluster_table.groupby('cluster')['target'].agg(blocks='size', anomaly_rate='mean').reset_index()
cluster_rates.to_csv(tables / 'kmeans_cluster_anomaly_rates.csv', index=False)
display(cluster_rates)

,cluster,blocks,anomaly_rate
0,0,327219,0.010598
1,1,5025,0.669453
2,2,8786,0.365695
3,3,4005,0.014232


## Summary

The descriptive statistics, correlations, PCA and clusters use training data only. The k-means elbow is gradual; k=4 is an illustrative choice. Clusters describe the data and are not evaluated as a detector. PCA fitted on all training blocks here is separate from the normal-only PCA detector in notebook 03.
